## Generar predicciones Langmuir Freundlich

In [ ]:
#!pipenv install colorama

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from lmfit import Model
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

#Sample
ce = np.array([0,0.0493, 0.0346, 0.0382, 0.0222, 0.0214, 0.0155, 0.0097, 0.0084, 0.0042])
qe = np.array([0, 0.0251, 0.0249, 0.0253, 0.0234, 0.0240, 0.0180, 0.0160, 0.0154, 0.0070])


def order_sample(ce, qe):
    pears = sorted(zip(ce, qe))
    x, y= zip(*pears)
    return list(x), list(y)


def langmuir(ce, qm, k):
    return (qm * k * ce) / (1 + k * ce)

def freundlich(ce, kf, n):
    return kf * (ce ** (1 / n))


def get_langmuir_params(ce, qm, k):
    model = Model(langmuir)
    params = model.make_params(qm=qm, k=k)
    result = model.fit(qe, params, ce=ce)
    print(result.summary())
    return result.best_fit,result.aic, result.bic, len(params)

def get_freundlich_params(ce,kf,n):
    model = Model(freundlich)
    params = model.make_params(kf=kf, n=n)
    result = model.fit(qe, params, ce=ce)
    return result.best_fit, result.aic, result.bic, len(params)


#ordeno sample
ce, qe =  order_sample(ce,qe)
    
# lagmuir parametros sacados de la linealizacion
qm = 0.0318
k = 96.7092
#pred
qe_langmuir_pred, lag_aic, lag_bic, lag_params_len = get_langmuir_params(ce,qm,k)


# frenund parametros sacados de la linealizacion
kf =  0.1291
n= 2.0885
#pred
qe_freundlich_pred, f_aic, f_bic, f_params_len = get_freundlich_params(ce,kf,n)


residuales_langmuir = qe - qe_langmuir_pred
residuales_freundlich = qe - qe_freundlich_pred

{'model': 'Model(langmuir)', 'method': 'leastsq', 'ndata': 10, 'nvarys': 2, 'nfree': 8, 'chisqr': 1.7671577028327766e-05, 'redchi': 2.2089471285409707e-06, 'aic': -128.46138119694177, 'bic': -127.85621101095369, 'rsquared': 0.9737088398133037, 'nfev': 13, 'max_nfev': 6000, 'aborted': False, 'errorbars': True, 'success': True, 'message': 'Fit succeeded.', 'lmdif_message': 'Both actual and predicted relative reductions in the sum of squares\n  are at most 0.000000', 'ier': 1, 'nan_policy': 'raise', 'scale_covar': True, 'calc_covar': True, 'ci_out': None, 'col_deriv': False, 'flatchain': None, 'call_kws': {'Dfun': None, 'full_output': 1, 'col_deriv': 0, 'ftol': 1.5e-08, 'xtol': 1.5e-08, 'gtol': 0.0, 'maxfev': 12000, 'epsfcn': 1e-10, 'factor': 100, 'diag': None}, 'var_names': ['qm', 'k'], 'user_options': None, 'kws': {}, 'init_values': {'qm': 0.0318, 'k': 96.7092}, 'best_values': {'qm': np.float64(0.032282483562522495), 'k': np.float64(97.23278024921449)}, 'params': [('qm', np.float64(0.03

In [ ]:
from colorama import Fore, Style

def metrics(results):
    for result in results:
        print(f"{Fore.CYAN}{'=' * 40}{Style.RESET_ALL}")
        print(f"{Fore.MAGENTA}Model: {result['model_name'].title()}{Style.RESET_ALL}")
        print(f"{Fore.CYAN}{'-' * 40}{Style.RESET_ALL}")
        
        print(f"{Fore.YELLOW}Statistics:{Style.RESET_ALL}")
        for key, value in result["statistics"].items():
            label = key.replace('_', ' ').title()
            print(f"  {Fore.GREEN}{label:<25}{Style.RESET_ALL}: {value:.4f}")
        
        print(f"\n{Fore.YELLOW}Residuals:{Style.RESET_ALL}")
        for key, value in result["residuals"].items():
            label = key.replace('_', ' ').title()
            if isinstance(value, (int, np.integer)):
                status = "Yes" if value == 1 else "No"
                print(f"  {Fore.GREEN}{label:<25}{Style.RESET_ALL}: {Fore.BLUE}{status}{Style.RESET_ALL}")
            else:
                print(f"  {Fore.GREEN}{label:<25}{Style.RESET_ALL}: {value:.4f}")
        print(f"{Fore.CYAN}{'=' * 40}{Style.RESET_ALL}\n")

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(ce, qe, label="Datos reales", color="black")
plt.plot(ce, qe_langmuir_pred, label="Langmuir", color="blue")
plt.plot(ce, qe_freundlich_pred, label="Freundlich", color="orange")
plt.title("Predicciones vs Datos Reales")
plt.xlabel("Ce")
plt.ylabel("Qe")
plt.legend()

# Comparacion

#### Obtener estadisticos

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from scipy import stats
from statsmodels.stats.stattools import durbin_watson
import math

def r_squared(y_exp, y_pred):
    '''Calculo de R2 resolviendo con la suma de cuadrado de los residuos y suma total de cuadrados.'''
    rss = np.sum((y_exp - y_pred) ** 2)
    y_mean = np.mean(y_exp)
    tss = np.sum((y_exp - y_mean) ** 2)
    r2 = 1 - rss / tss
    return r2

def linear_r_squared(pearson_coef):
    '''Calculo de R2 cuando se trata de un modelo lineal.'''
    return pearson_coef ** 2

def adjust_r_squared(y_exp, y_pred, num_params):
    '''Calcula el R² ajustado que penaliza el número de parámetros del modelo.'''
    n = len(y_exp)
    r2_adjusted = 1 - (((n - 1) / (n - num_params - 1)) * (1 - r_squared(y_exp, y_pred)))
    return r2_adjusted

def chi_squared(y_exp, y_pred):
    '''Realiza el cálculo matemático utilizando la función chisquare de scipy.'''
    chi_squared_value = np.sum(((y_exp - y_pred) ** 2) / ((y_pred + 1e-10) ** 2))
    return chi_squared_value

def adjust_chi_squared(y_exp, y_pred, num_params):
    '''Calcula el chi-cuadrado ajustado que penaliza el número de parámetros del modelo.'''
    n = len(y_exp)
    chi2_adjusted = chi_squared(y_exp, y_pred) / (n - num_params)
    return chi2_adjusted

def rmse(y_exp, y_pred):
    '''Calcula la raíz del error cuadrático medio (RMSE).'''
    return np.sqrt(mean_squared_error(y_exp, y_pred))

def sse(y_exp, y_pred):
    '''Calcula la suma de los errores al cuadrado (SSE).'''
    return np.sum((y_exp - y_pred) ** 2)

def hybrid(y_exp, y_pred, num_params):
    '''Calcula un indicador híbrido que combina los errores cuadrados con la cantidad de parámetros.'''
    n = len(y_exp)
    if n > 1:
        hybrid_value = (100 / (n - num_params)) * np.sum((y_exp - y_pred) ** 2 / y_exp)
        return hybrid_value
    return None
    
def check_residuals(residuals):
    '''Evalúa los residuos del modelo para comprobar tres supuestos importantes.'''
    # Normalidad (Shapiro-Wilk)
    _, normality_p = stats.shapiro(residuals)

    # Homocedasticidad (Levene)
    _, homo_p = stats.levene(residuals, np.ones_like(residuals))

    # Autocorrelación (Durbin-Watson)
    dw_stat = durbin_watson(residuals)

    return {
        'normality_pvalue': normality_p,
        'homoscedasticity_pvalue': homo_p,
        'durbin_watson': dw_stat,
        'passes_normality': 0 if normality_p > 0.05 else 1, #distribucion normal residuos
        'passes_homoscedasticity': 0 if homo_p > 0.05 else 1, #varianza constante
        'passes_independence': 1 if  (1.5 < dw_stat < 2.5) else 0 #que no esten autocorrelacionados
    }

In [ ]:
def all_statistics(y_exp, y_pred, num_params, aic, bic):
    '''Calcula todas las estadísticas de ajuste del modelo.'''
    hybrid_value = hybrid(y_exp, y_pred, num_params)
    stats_dict = {
        "r_squared": round(r_squared(y_exp, y_pred), 4),
        "adjust_r_squared": round(adjust_r_squared(y_exp, y_pred, num_params), 4),
        "chi_squared": round(chi_squared(y_exp, y_pred), 4),
        "adjust_chi_squared": round(adjust_chi_squared(y_exp, y_pred, num_params), 4),
        "RMSE": round(rmse(y_exp, y_pred), 4),
        "SSE": round(sse(y_exp, y_pred), 4),
        "HYBRID": round(hybrid_value, 4) if hybrid_value else None,
        "AIC": round(aic, 4),
        "BIC": round(bic, 4)
    }
    return stats_dict

In [ ]:
results = []

stats_lag = {
    "model_name": "Langmuir",
    "statistics": all_statistics(qe,qe_langmuir_pred, lag_params_len, aic=lag_aic, bic=lag_bic ),
    "residuals": check_residuals(residuales_langmuir)
}

stats_f = {
    "model_name": "freundlich",
    "statistics": all_statistics(qe,qe_freundlich_pred, f_params_len, f_aic, f_bic),
    "residuals": check_residuals(residuales_freundlich)
}

results.append(stats_lag)
results.append(stats_f)

### Comparacion con heuristica que contempla ponderacion de errores y analisis de residuos

In [ ]:
def get_best_model(results, key):
    scores = {}

    for result in results:

        rmse = result['statistics'].get('RMSE', float('inf'))
        aic = result['statistics'].get('AIC', float('inf'))
        chi_squared = result['statistics'].get('chi_squared', float('inf'))
        r_squared_adjusted = result['statistics'].get('r_squared_adjusted', 0)
        durbin_watson = result['statistics'].get('durbin_watson', 0)

        passes_normality = result['residuals'].get('passes_normality', False)
        passes_homoscedasticity = result['residuals'].get('passes_homoscedasticity', False)
        passes_independence = result['residuals'].get('passes_independence', False)


        if not math.isfinite(r_squared_adjusted) or not math.isfinite(rmse) or not math.isfinite(aic):
            print(f"Advertencia: Datos inválidos en modelo {result[key]}. Se omitirán del cálculo.")
            continue


        score = (
                max(r_squared_adjusted, 0) * 0.30 +  # Asegurar que r_squared no sea negativo
                (1 / max(rmse, 1e-9)) * 0.3 +  # Evitar división por 0
                (1 / max(abs(aic), 1e-9)) * 0.25 +  # Asegurar que AIC no sea 0
                (1 / (1 + chi_squared)) * 0.1 +
                (0.05 if passes_normality else 0) +  # Normalidad
                (0.05if passes_homoscedasticity else 0) +  # Homocedasticidad
                (0.05 if passes_independence else 0)  # Independencia
        )


        scores[result[key]] = score

    if not scores:
        raise ValueError("No se pudieron calcular puntajes válidos para ningun modelo.")

    best_model = max(scores, key=scores.get)
    return best_model

In [ ]:
get_best_model(results, "model_name")

In [ ]:
metrics(results)

### Comparacion usando modelos 

In [ ]:
def best_alpha():
    ridge = Ridge()
    param_grid = {'alpha': [0.001, 0.1, 0.05,0.2, 1, 5, 10, 100, 1000]}
    grid_search = GridSearchCV(estimator=ridge, param_grid=param_grid, scoring='neg_mean_squared_error', cv=5)
    grid_search.fit(X_scaled, y)
    return grid_search.best_params_['alpha']


#creo matriz
X = np.column_stack((qe_langmuir_pred, qe_freundlich_pred))
y = qe

# estandarizo para escalar las magnitudes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#elijo best alpha
alpha =  best_alpha()

# regresion
ridge = Ridge(alpha=alpha)
ridge.fit(X_scaled, y)
ridge_coefs = ridge.coef_
qe_pred_ridge = ridge.predict(X_scaled)

# estadisticos
r2_ridge = r2_score(y, qe_pred_ridge)
mse_ridge = mean_squared_error(y, qe_pred_ridge)
mae_ridge = mean_absolute_error(y, qe_pred_ridge)

if r2_ridge > 0.7:
    print("Ridge es suficiente para predecir los datos.")
else:
    mlp = MLPRegressor(hidden_layer_sizes=(1,), max_iter=500, random_state=42)
    mlp.fit(X, y)
    y_pred_mlp = mlp.predict(X)
    r2_mlp = r2_score(y, y_pred_mlp)

    print("\n MLP")
    print(f"R^2 para Red Neuronal: {r2_mlp:.4f}")

    if r2_mlp > r2_ridge:
        print("Red neuronal mejor q ridge")
    else:
        print("No ayuda la red neuronal, cambiar algo en los modelos")

residuales_ridge = qe - qe_pred_ridge

# graficos
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(ce, qe, label="Datos reales", color="black")
plt.plot(ce, qe_pred_ridge, label="Ridge", color="red")
plt.plot(ce, qe_langmuir_pred, label="Langmuir", color="blue")
plt.plot(ce, qe_freundlich_pred, label="Freundlich", color="orange")
plt.title("Predicciones vs Datos Reales")
plt.xlabel("Ce")
plt.ylabel("Qe")
plt.legend()


plt.subplot(1, 2, 2)
plt.scatter(ce, qe, label="Datos reales", color="black")
plt.scatter(ce, qe_langmuir_pred, label="Predicciones Langmuir", color="blue")
plt.scatter(ce, qe_freundlich_pred, label="Predicciones Freundlich", color="orange")
plt.title("Predicciones vs Datos Reales")
plt.xlabel("Datos Reales (Qe)")
plt.ylabel("Predicciones")
plt.legend()
plt.show()

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))


plt.subplot(1, 2, 1)
plt.scatter(qe_langmuir_pred, residuales_langmuir, label="Langmuir", color="blue")
plt.title("Residuos vs Predicciones (Langmuir)")
plt.xlabel("Predicciones (Langmuir)")
plt.ylabel("Residuos")
plt.axhline(0, color="black", linestyle="--", linewidth=0.8)
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(qe_freundlich_pred, residuales_freundlich, label="Freundlich", color="orange")
plt.title("Residuos vs Predicciones (Freundlich)")
plt.xlabel("Predicciones (Freundlich)")
plt.ylabel("Residuos")
plt.axhline(0, color="black", linestyle="--", linewidth=0.8)
plt.legend()

plt.tight_layout()
plt.show()

# Resultados
print("--- Resultados ---")
print(f"Best alpha: {alpha}")
print(f"Coeficiente Langmuir (Ridge): {ridge_coefs[0]}")
print(f"Coeficiente Freundlich (Ridge): {ridge_coefs[1]}")
print(f"R^2 para Ridge: {r2_ridge:.4f}")
print(f"MSE para Ridge: {mse_ridge:.6f}")
print(f"MAE para Ridge: {mae_ridge:.6f}")

In [ ]:
print(X_scaled)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(qe_pred_ridge, residuales_ridge)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicciones Ridge')
plt.ylabel('Residuos')
plt.title('Análisis de Residuos - Modelo Ridge')
plt.show()